In [1]:
# Membuat spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Tugas4-PengenalanSpark") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

26/09/17 05:25:54 WARN Utils: Your hostname, fais resolves to a loopback address: 127.0.1.1; using 10.176.17.43 instead (on interface wlp3s0)
26/09/17 05:25:54 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/17 05:25:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# Sel ini membuat dataset baru untuk Tugas Mandiri Pertemuan 4 dan mengunggahnya ke HDFS
import numpy as np
import pandas as pd

np.random.seed(99)
n = 1000
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga", "Olahraga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo", "Kebumen"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-09-01", "2026-09-30", freq="D")

data = {
    "order_id": [f"ORD-{3000 + i}" for i in range(n)],
    "tanggal": np.random.choice(tanggal_range, size=n).astype(str),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 12, size=n),
    "harga_satuan": np.random.choice([20000, 45000, 60000, 90000, 125000, 200000, 350000], size=n),
    "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    "rating": np.random.choice([1, 2, 3, 4, 5, np.nan], size=n, p=[0.03, 0.02, 0.10, 0.30, 0.35, 0.20]),
}
df_tugas4 = pd.DataFrame(data)
df_tugas4.to_csv("transaksi_september_2026.csv", index=False)
print(f"Dataset dibuat: {df_tugas4.shape[0]} baris")

# Mengunggah ke HDFS
!hdfs dfs -mkdir -p /user/mahasiswa/tugas4
!hdfs dfs -put -f transaksi_september_2026.csv /user/mahasiswa/tugas4/
print("Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv")

Dataset dibuat: 1000 baris
Berhasil diunggah ke HDFS: /user/mahasiswa/tugas4/transaksi_september_2026.csv


In [3]:
# A. Membaca dan Eksplorasi Awal
# Membaca CSV LANGSUNG dari HDFS menggunakan Spark — perhatikan prefix "hdfs://"
df = spark.read.csv("hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026.csv",header=True, inferSchema=True)
df.printSchema()
print("Jumlah baris dari HDFS:", df.count())
df.show(10)

root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- rating: double (nullable = true)

Jumlah baris dari HDFS: 1000
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|order_id|            tanggal|            kategori|      kota|unit_terjual|harga_satuan|metode_pembayaran|rating|
+--------+-------------------+--------------------+----------+------------+------------+-----------------+------+
|ORD-3000|2026-09-02 00:00:00|        Rumah Tangga|Yogyakarta|           3|       90000|              COD|   4.0|
|ORD-3001|2026-09-04 00:00:00|   Makanan & Minuman|      Solo|           3|      200000|         E-Wallet|   5.0|
|ORD-3002|2026-09-26 00:00:00|Kesehatan & Kecan...|  Semaran

In [4]:
# B. Menangani Data Kosong 
from pyspark.sql.functions import col
# Menghitung data kosong
jumlah_Null = df.filter(col("rating").isNull()).count()
print(f"Jumlah data rating yang NULL: {jumlah_Null}")

# Menangani data kosong dengan mengisi angka 0
df_null = df.na.fill(0, subset=["rating"])

# Mengecek data kosong
df_cekNull = df_null.filter(col("rating").isNull()).count()
print(f"Jumlah data rating yang NULL: {df_cekNull}")

Jumlah data rating yang NULL: 204
Jumlah data rating yang NULL: 0


In [5]:
# C. Transformasi Data
from pyspark.sql.functions import when
# Menambahkan kolom baru: total_pendapatan = unit_terjual x harga_satuan
df_transform = df_null.withColumn("total_pendapatan", col("unit_terjual") * col("harga_satuan"))\
                      .withColumn("tier_transaksi" ,when(col("total_pendapatan") > 500000, "Besar").otherwise("Kecil"))
df_transform.select("order_id", "total_pendapatan", "tier_transaksi").show()

+--------+----------------+--------------+
|order_id|total_pendapatan|tier_transaksi|
+--------+----------------+--------------+
|ORD-3000|          270000|         Kecil|
|ORD-3001|          600000|         Besar|
|ORD-3002|          480000|         Kecil|
|ORD-3003|         2100000|         Besar|
|ORD-3004|          600000|         Besar|
|ORD-3005|          100000|         Kecil|
|ORD-3006|           40000|         Kecil|
|ORD-3007|          720000|         Besar|
|ORD-3008|          140000|         Kecil|
|ORD-3009|          900000|         Besar|
|ORD-3010|         1400000|         Besar|
|ORD-3011|          240000|         Kecil|
|ORD-3012|         1000000|         Besar|
|ORD-3013|         1600000|         Besar|
|ORD-3014|           60000|         Kecil|
|ORD-3015|         1600000|         Besar|
|ORD-3016|          450000|         Kecil|
|ORD-3017|         1000000|         Besar|
|ORD-3018|          990000|         Besar|
|ORD-3019|          250000|         Kecil|
+--------+-

In [6]:
# D. Analisis GroupBy
from pyspark.sql.functions import sum as spark_sum, count, avg
# 1. Kategori apa yang memiliki total_pendapatan tertinggi?
df_transform.groupBy("kategori").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
).orderBy(col("total_pendapatan").desc()).show(1)

# 2. Kota mana dengan jumlah tier transaksi "Besar" terbanyak
df_transform.filter(col("tier_transaksi") == "Besar").groupBy("kota")\
    .agg(count("order_id").alias("tier_Besar")).orderBy(col("tier_Besar").desc()).show(1)

# 3. Rata-rata rating untuk masing-masing metode_pembayaran
df_transform.groupBy("metode_pembayaran")\
    .agg(avg("rating").alias("rata_rata_rating")).orderBy(col("rata_rata_rating").desc()).show()

+------------+----------------+
|    kategori|total_pendapatan|
+------------+----------------+
|Rumah Tangga|       138665000|
+------------+----------------+
only showing top 1 row

+----+----------+
|kota|tier_Besar|
+----+----------+
|Solo|        92|
+----+----------+
only showing top 1 row

+-----------------+------------------+
|metode_pembayaran|  rata_rata_rating|
+-----------------+------------------+
|              COD|3.3745019920318726|
|    Transfer Bank|3.3399209486166006|
|         E-Wallet|             3.292|
|     Kartu Kredit|3.1910569105691056|
+-----------------+------------------+



In [7]:
# E. Menyimpan Hasil ke HDFS
# 1. Tentukan path folder penyimpanan (Hapus akhiran .csv agar rapi)
hasil_data = "hdfs://localhost:9000/user/mahasiswa/tugas4/transaksi_september_2026"

# 2. Simpan DataFrame hasil olahan ke HDFS
df_transform.write.csv(hasil_data, header=True, mode="overwrite")

# 3. Verifikasi direktori dan partisi berkas (Pastikan nama foldernya persis sama!)
!hdfs dfs -ls /user/mahasiswa/tugas4/transaksi_september_2026

Found 2 items
-rw-r--r--   3 fais supergroup          0 2026-09-17 05:26 /user/mahasiswa/tugas4/transaksi_september_2026/_SUCCESS
-rw-r--r--   3 fais supergroup      97296 2026-09-17 05:26 /user/mahasiswa/tugas4/transaksi_september_2026/part-00000-57c2583f-597b-4945-87c4-e95f0df67c4f-c000.csv
